# 03b - Baseline Models
Trenink tri baseline klasifikatoru se SMOTE na nevyrovnanem datasetu.
Vystupy: `results/baseline_results.pkl`, `results/baseline_results_wo_gpa.pkl`, `models/baseline/*.pkl`

In [1]:
import joblib
import warnings
import os
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

warnings.filterwarnings('ignore')

X_train_prep, X_test_prep, y_train, y_test = joblib.load('../data/processed/split_data.pkl')

import matplotlib as _mpl

# Paul Tol bright palette — colorblind-safe, perceptually uniform
_PALETTE = ['#0077BB', '#EE7733', '#009988', '#CC3311', '#33BBEE', '#EE3377', '#BBBBBB']

_mpl.rcParams.update({
    # Color cycle
    'axes.prop_cycle':       _mpl.cycler(color=_PALETTE),

    # Backgrounds
    'figure.facecolor':      'white',
    'axes.facecolor':        '#F8F9FA',   # subtle off-white — "card" feel

    # Spines
    'axes.edgecolor':        '#DEE2E6',
    'axes.linewidth':        0.9,
    'axes.spines.top':       False,
    'axes.spines.right':     False,

    # Grid — solid, very light
    'axes.grid':             True,
    'axes.axisbelow':        True,
    'grid.color':            '#E9ECEF',
    'grid.linestyle':        '-',
    'grid.linewidth':        0.8,
    'grid.alpha':            1.0,

    # Typography
    'font.family':           'sans-serif',
    'font.sans-serif':       ['Helvetica Neue', 'Arial', 'DejaVu Sans'],
    'axes.titlesize':        14,
    'axes.titleweight':      'bold',
    'axes.titlepad':         14,
    'axes.labelsize':        12,
    'axes.labelweight':      'regular',
    'xtick.labelsize':       10,
    'ytick.labelsize':       10,
    'legend.fontsize':       10,
    'legend.title_fontsize': 11,
    'figure.titlesize':      16,
    'figure.titleweight':    'bold',

    # Text colors — near-black for contrast
    'text.color':            '#212529',
    'axes.labelcolor':       '#495057',
    'xtick.color':           '#495057',
    'ytick.color':           '#495057',

    # Ticks
    'xtick.direction':       'out',
    'ytick.direction':       'out',
    'xtick.major.size':      4,
    'ytick.major.size':      4,
    'xtick.minor.visible':   False,
    'ytick.minor.visible':   False,
    'xtick.major.pad':       5,
    'ytick.major.pad':       5,

    # Lines & markers
    'lines.linewidth':       2.0,
    'lines.markersize':      7,
    'patch.linewidth':       0.6,

    # Legend
    'legend.frameon':        True,
    'legend.framealpha':     0.92,
    'legend.edgecolor':      '#DEE2E6',
    'legend.fancybox':       True,
    'legend.borderpad':      0.6,

    # Figure / saving
    'figure.dpi':            100,
    'figure.figsize':        [8, 5],
    'savefig.dpi':           150,
    'savefig.bbox':          'tight',
    'savefig.facecolor':     'white',
    'savefig.edgecolor':     'none',
})


## Definice modelu

In [2]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        random_state=42
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ),
}

# Modely vyzadujici skálovani (pro referenci v downstream noteboocich)
scaled_models = {'Logistic Regression'}

In [3]:
## Trenink se SMOTE

In [4]:
smote   = SMOTE(random_state=42)
results = {}

for name, model in models.items():
    X_tr_resampled, y_train_resampled = smote.fit_resample(X_train_prep, y_train)
    model.fit(X_tr_resampled, y_train_resampled)

    y_pred  = model.predict(X_test_prep)
    y_proba = model.predict_proba(X_test_prep)[:, 1]

    results[name] = {
        'model':    model,
        'accuracy': accuracy_score(y_test, y_pred),
        'roc_auc':  roc_auc_score(y_test, y_proba),
        'report':   classification_report(y_test, y_pred),
        'y_pred':   y_pred,
    }

    print(f"{'=' * 50}")
    print(f"  {name}")
    print(f"{'=' * 50}")
    print(f"  Accuracy : {results[name]['accuracy']:.4f}")
    print(f"  ROC-AUC  : {results[name]['roc_auc']:.4f}")
    print(f"\n{results[name]['report']}")

  Logistic Regression
  Accuracy : 0.7480
  ROC-AUC  : 0.7981

              precision    recall  f1-score   support

           0       0.89      0.76      0.82      1529
           1       0.48      0.69      0.56       471

    accuracy                           0.75      2000
   macro avg       0.68      0.73      0.69      2000
weighted avg       0.79      0.75      0.76      2000

  Random Forest
  Accuracy : 0.7795
  ROC-AUC  : 0.7972

              precision    recall  f1-score   support

           0       0.87      0.84      0.85      1529
           1       0.53      0.59      0.56       471

    accuracy                           0.78      2000
   macro avg       0.70      0.71      0.71      2000
weighted avg       0.79      0.78      0.78      2000

  Gradient Boosting
  Accuracy : 0.7920
  ROC-AUC  : 0.8009

              precision    recall  f1-score   support

           0       0.85      0.88      0.87      1529
           1       0.57      0.49      0.53       471

 

## Ablacni studie: vliv odstraneni GPA priznaku
Overuje, zda modely nespolehaji prilis na GPA featury, ktere nemusi byt vzdy dostupne.

In [5]:
cols_to_drop_gpa = ['CGPA', 'GPA_trend']
X_train_wo_gpa   = X_train_prep.drop(columns=cols_to_drop_gpa, errors='ignore')
X_test_wo_gpa    = X_test_prep.drop(columns=cols_to_drop_gpa,  errors='ignore')

results_wo_gpa = {}

for name, model in models.items():
    X_tr_resampled, y_train_resampled = smote.fit_resample(X_train_wo_gpa, y_train)
    model.fit(X_tr_resampled, y_train_resampled)

    y_pred  = model.predict(X_test_wo_gpa)
    y_proba = model.predict_proba(X_test_wo_gpa)[:, 1]

    results_wo_gpa[name] = {
        'model':    model,
        'accuracy': accuracy_score(y_test, y_pred),
        'roc_auc':  roc_auc_score(y_test, y_proba),
        'report':   classification_report(y_test, y_pred),
    }

    print(f"\n{'=' * 50}")
    print(f"  {name} (Bez GPA)")
    print(f"{'=' * 50}")
    print(f"  Accuracy : {results_wo_gpa[name]['accuracy']:.4f}")
    print(f"  ROC-AUC  : {results_wo_gpa[name]['roc_auc']:.4f}")
    print(f"\n{results_wo_gpa[name]['report']}")


  Logistic Regression (Bez GPA)
  Accuracy : 0.6640
  ROC-AUC  : 0.6908

              precision    recall  f1-score   support

           0       0.84      0.69      0.76      1529
           1       0.36      0.58      0.45       471

    accuracy                           0.66      2000
   macro avg       0.60      0.63      0.60      2000
weighted avg       0.73      0.66      0.69      2000


  Random Forest (Bez GPA)
  Accuracy : 0.7585
  ROC-AUC  : 0.6919

              precision    recall  f1-score   support

           0       0.80      0.91      0.85      1529
           1       0.48      0.28      0.36       471

    accuracy                           0.76      2000
   macro avg       0.64      0.59      0.60      2000
weighted avg       0.73      0.76      0.73      2000


  Gradient Boosting (Bez GPA)
  Accuracy : 0.7780
  ROC-AUC  : 0.7033

              precision    recall  f1-score   support

           0       0.80      0.95      0.87      1529
           1       0.58

## Ulozeni vysledku a modelu

In [6]:
os.makedirs('../results',         exist_ok=True)
os.makedirs('../models/baseline', exist_ok=True)

# Vysledky pro 033
joblib.dump(results,        '../results/baseline_results.pkl')
joblib.dump(results_wo_gpa, '../results/baseline_results_wo_gpa.pkl')
print("Vysledky ulozeny do results/")

# Individualni baseline modely
for name, res in results.items():
    safe_name = name.lower().replace(' ', '_')
    joblib.dump(res['model'], f'../models/baseline/{safe_name}.pkl')
    print(f"  models/baseline/{safe_name}.pkl")

print("\nBaseline modely ulozeny do models/baseline/")

# Decision Tree ulozime zvlast — potrebujeme ho pro vizualizaci v 06f
joblib.dump(results['Decision Tree']['model'], '../models/baseline/decision_tree.pkl')
print("  models/baseline/decision_tree.pkl (pro XAI vizualizaci)")

Vysledky ulozeny do results/
  models/baseline/logistic_regression.pkl
  models/baseline/random_forest.pkl
  models/baseline/gradient_boosting.pkl
  models/baseline/decision_tree.pkl

Baseline modely ulozeny do models/baseline/
  models/baseline/decision_tree.pkl (pro XAI vizualizaci)
